# Боти, смерть, прориви

Порівняння стратегій ботів Sim.Cli (tech 6.1–6.2, критерій етапу 1: «звіти показують різницю між стратегіями ботів»). Дані — `sim run --bot`, один рядок JSONL на забіг; спільний завантажувач — `simruns.py`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from simruns import BOTS, label, load, ranks

RUNS, SEED_FROM = 1000, 1  # per bot; data regenerates by itself when Core, Sim.Cli or content changes
data = load(runs=RUNS, seed_from=SEED_FROM)
runs = data.runs
pd.options.display.float_format = "{:.2f}".format
plt.rcParams["figure.figsize"] = (10, 4)


def named(df, axis=0):
    """Replaces content IDs in the index (axis=0) or columns (axis=1) with Ukrainian names."""
    return df.rename(index=label) if axis == 0 else df.rename(columns=label)


def per_run(long, value, by):
    """Mean of value per run for each bot, counting runs where it never happened as zero."""
    total = long.pivot_table(index=by, columns="bot", values=value, aggfunc="sum", fill_value=0)
    return total.div(runs.groupby("bot").size(), axis=1).reindex(columns=list(BOTS), fill_value=0)


## Підсумок за ботами

Вік смерті в роках, середній досягнутий етап і частки причин кінця забігу.

In [ ]:
age = runs.groupby("bot").death_age_years
summary = pd.DataFrame({
    "забігів": runs.groupby("bot").size(),
    "вік смерті, медіана": age.median(),
    "p10": age.quantile(0.1),
    "p90": age.quantile(0.9),
    "етап, середній": runs.groupby("bot").final_stage.mean(),
    "спроб прориву": runs.groupby("bot").breakthrough_attempts.mean(),
}).reindex(list(BOTS))
causes = pd.crosstab(runs.bot, runs.end_cause, normalize="index").reindex(list(BOTS))
summary.join(named(causes, axis=1))

In [ ]:
fig, ax = plt.subplots()
for bot in BOTS:
    ax.hist(runs.loc[runs.bot == bot, "death_age_years"], bins=40, histtype="step", linewidth=1.5, label=bot)
ax.set(xlabel="вік смерті, роки", ylabel="забігів", title="Розподіл віку смерті")
ax.legend();

## Досягнутий етап

Частка забігів, що закінчились на кожному етапі ці.

In [ ]:
pd.crosstab(runs.bot, runs.final_stage, normalize="index").reindex(list(BOTS))

## Вік на кожному етапі

Вік успішного прориву на етап *N* (тобто з етапу *N − 1*) і частка забігів, що до нього дійшли.

In [ ]:
bt = data.breakthroughs
success = bt[bt.result == "success"].assign(stage=lambda d: d.from_stage + 1)
reached = success.groupby(["bot", "stage"]).seed.nunique().div(runs.groupby("bot").size(), level="bot")
stage_age = success.groupby(["bot", "stage"]).age_years.describe(percentiles=[0.1, 0.5, 0.9])
stage_age[["count", "10%", "50%", "90%"]].assign(дійшли=reached)

In [ ]:
stages = sorted(success.stage.unique())
fig, axes = plt.subplots(1, len(stages), sharey=True, figsize=(4 * len(stages), 4))
for ax, stage in zip(axes if len(stages) > 1 else [axes], stages):
    at = success[success.stage == stage]
    ax.boxplot([at.loc[at.bot == b, "age_years"] for b in BOTS], tick_labels=BOTS)
    ax.set(title=f"Прорив на етап {stage}", ylabel="вік, роки")

## Результати спроб прориву

Частки успіху, провалу й загибелі серед усіх спроб, за етапом, з якого прориваються.

In [ ]:
pd.crosstab([bt.bot, bt.from_stage], bt.result, normalize="index")

## Якість прориву

Розподіл рангів успішних проривів (від гіршого до кращого) і середній ранг залежно від віку.

In [ ]:
order = ranks()
rank_share = pd.crosstab([success.bot, success.stage], success["rank"], normalize="index")
named(rank_share.reindex(columns=[r for r in order if r in rank_share.columns]), axis=1)

In [ ]:
success = success.assign(rank_index=success["rank"].map({r: i for i, r in enumerate(order)}),
                         age_bin=(success.age_years // 5) * 5)
fig, ax = plt.subplots()
for bot in BOTS:
    by_age = success[success.bot == bot].groupby("age_bin").rank_index.mean()
    ax.plot(by_age.index, by_age.values, marker="o", label=bot)
ax.set(xlabel="вік прориву, роки (кошики по 5)", ylabel="середній ранг", title="Якість прориву залежно від віку")
ax.set_yticks(range(len(order)), [label(r) for r in order])
ax.legend();

In [ ]:
fig, ax = plt.subplots()
for result, marker in [("success", "o"), ("failed", "x"), ("death", "^")]:
    at = bt[bt.result == result]
    ax.scatter(at.age_years, at.foundation / at.threshold, s=6, marker=marker, alpha=0.4, label=result)
ax.axhline(1, color="grey", linewidth=0.8)
ax.set(xlabel="вік спроби, роки", ylabel="фундамент / поріг", title="Запас фундаменту на момент спроби")
ax.legend();

## Розподіл часу

Частка тижнів забігу за заняттями (середнє за ботом); `event` — тижні, які забирають події.

In [ ]:
weeks = per_run(data.activity_weeks, "weeks", "activity")
shares = (weeks / runs.groupby("bot").weeks.mean()).T
named(shares, axis=1)

In [ ]:
named(shares, axis=1).plot.barh(stacked=True, figsize=(10, 3), title="Частки часу за заняттями")
plt.legend(bbox_to_anchor=(1, 1), loc="upper left", fontsize="small");